# Triagem Automática de Demandas Urbanas — Aracaju/SE

**Fundamentos da Inteligência Artificial (COMP0506) — UFS 2026.2**

Classificador de reclamações de cidadãos em 4 secretarias, usando **Naive Bayes** como modelo baseline.

Roteiro do notebook:
1. Pré-processamento do texto (demonstração passo a passo)
2. Carregamento e verificação do dataset
3. Separação treino/teste
4. Treinamento do modelo
5. Métricas de avaliação (acurácia, precision, recall, F1, matriz de confusão)
6. Validação cruzada
7. Análise dos erros
8. Teste com frases novas, fora do dataset
9. Demonstração: classificar um chamado

## 1. Pré-processamento do texto — demonstração passo a passo

As células abaixo mostram, com uma frase de exemplo, cada etapa da limpeza: minúsculas, remoção de pontuação, separação em palavras (tokens) e remoção de *stopwords*.

In [ ]:
# Transformando todo o texto em minúsculo.
texto = "TEM um BURACO gigante na Avenida Hermes Fontes!!"
texto_minusculo = texto.lower()

print(texto_minusculo)

In [ ]:
import string

# Tirando toda pontuação do texto.
texto_sem_pontuacao = texto_minusculo.translate(str.maketrans("", "", string.punctuation))

print(texto_sem_pontuacao)

In [ ]:
# Separando o texto em tokens.
palavras = texto_sem_pontuacao.split()

print(palavras)


In [ ]:
# Filtrando as palavras úteis.
stop_words = ["tem", "um", "na", "no", "de", "do", "da"]
texto_limpo = [palavra for palavra in palavras if palavra not in stop_words]
print(texto_limpo)


In [ ]:
# Juntado as palavras úteis para formar o texto padronizado.
texto_padronizado = " ".join(texto_limpo)
print(texto_padronizado)

### Juntando as etapas em uma função

A função `limpar_texto` faz todas as etapas acima de uma vez. Ela é usada **dentro do pipeline do modelo**, então toda frase — de treino, de teste ou nova — passa pela mesma limpeza.

In [ ]:
import string

# Lista de palavras muito comuns que não ajudam a decidir a categoria
stop_words = ["tem", "um", "uma", "na", "no", "de", "do", "da", "a", "o", "e", "que",
              "em", "os", "as", "dos", "das", "para", "por", "com", "ao", "à"]

def limpar_texto(texto):
    texto = texto.lower()                                                  # 1. minúsculas
    texto = texto.translate(str.maketrans("", "", string.punctuation))     # 2. sem pontuação
    palavras = texto.split()                                               # 3. tokens
    palavras_uteis = [p for p in palavras if p not in stop_words]          # 4. sem stopwords
    return " ".join(palavras_uteis)                                        # 5. texto padronizado

print(limpar_texto("TEM um BURACO gigante na Avenida Hermes Fontes!!"))

## 2. Carregamento e verificação do dataset

In [ ]:
import pandas

data_frame = pandas.read_csv("chamados_municipais.csv")

print("Total de chamados:", len(data_frame))
print("Frases repetidas:", data_frame["Texto"].duplicated().sum())
print("Valores vazios:", data_frame.isna().sum().sum())
print()
print("Chamados por categoria:")
print(data_frame["Categoria"].value_counts())

In [ ]:
# Alguns exemplos do dataset
data_frame.sample(8, random_state=1)

## 3. Separação treino/teste

O modelo é avaliado em frases que **ele não viu no treino**. Usamos 80% dos dados para treinar e 20% para testar.

- `stratify=y`: mantém a mesma proporção de cada categoria no treino e no teste.
- `random_state=42`: fixa o sorteio, para o resultado ser sempre o mesmo ao rodar de novo.

In [ ]:
from sklearn.model_selection import train_test_split

x = data_frame["Texto"]
y = data_frame["Categoria"]

x_treino, x_teste, y_treino, y_teste = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

print("Frases de treino:", len(x_treino))
print("Frases de teste:", len(x_teste))
print("Frases do teste que também estão no treino:", x_teste.isin(set(x_treino)).sum())

## 4. Treinamento do modelo (pipeline)

O pipeline tem duas etapas:
1. **CountVectorizer** — aplica `limpar_texto` e transforma cada frase em uma contagem de palavras.
2. **MultinomialNB** (Naive Bayes) — calcula, a partir dessas contagens, a probabilidade de a frase pertencer a cada categoria.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline

modelo_baseline = make_pipeline(
    CountVectorizer(preprocessor=limpar_texto),
    MultinomialNB()
)

modelo_baseline.fit(x_treino, y_treino)

print("Modelo treinado.")
print("Tamanho do vocabulário:", len(modelo_baseline.named_steps["countvectorizer"].vocabulary_), "palavras")

## 5. Métricas de avaliação

- **Acurácia**: porcentagem de chamados classificados corretamente.
- **Precision**: dos chamados que o modelo mandou para uma secretaria, quantos eram mesmo dela.
- **Recall**: dos chamados que eram de uma secretaria, quantos o modelo encontrou.
- **F1-Score**: média harmônica entre precision e recall.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

previsoes = modelo_baseline.predict(x_teste)

print(f"Acurácia no teste: {accuracy_score(y_teste, previsoes):.1%}")
print()
print(classification_report(y_teste, previsoes, digits=3))

### Matriz de confusão

Cada linha é a categoria **real** e cada coluna é a categoria **prevista**. A diagonal mostra os acertos; fora dela estão as confusões entre secretarias.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_teste, previsoes, cmap="Blues", xticks_rotation=30, ax=ax
)
ax.set_title("Matriz de confusão — conjunto de teste")
ax.set_xlabel("Categoria prevista")
ax.set_ylabel("Categoria real")
plt.tight_layout()
plt.show()

## 6. Validação cruzada

Um único sorteio de treino/teste pode dar sorte ou azar. Na validação cruzada (5 partes), o dataset é dividido em 5 pedaços e o modelo é treinado e testado 5 vezes, cada vez com um pedaço diferente como teste.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

divisao = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
modelo_cv = make_pipeline(CountVectorizer(preprocessor=limpar_texto), MultinomialNB())

notas = cross_val_score(modelo_cv, x, y, cv=divisao, scoring="accuracy")

print("Acurácia em cada rodada:", [f"{n:.1%}" for n in notas])
print(f"Média: {notas.mean():.1%}  (desvio padrão: {notas.std():.1%})")

## 7. Análise dos erros

Olhar os erros mostra **por que** o modelo se confunde.

In [ ]:
pandas.set_option("display.max_colwidth", 120)

erros = pandas.DataFrame({
    "Texto": x_teste,
    "Categoria real": y_teste,
    "Categoria prevista": previsoes
})
erros = erros[erros["Categoria real"] != erros["Categoria prevista"]]

print(f"{len(erros)} erros em {len(x_teste)} chamados de teste")
erros

**Padrões observados nos erros:**
- Frases que **negam** um problema ("o asfalto está inteiro, o que precisa é...") confundem o modelo: ele conta a palavra "asfalto" e não entende o "não". O Naive Bayes não considera a ordem nem o sentido das palavras.
- Palavras que aparecem em mais de uma secretaria ("semáforo", "faixa", "luz", "parada") puxam para a categoria errada.
- Termos técnicos raros ("semafórico", "refletor") ou gírias ("barriga na pista") aparecem pouco no treino.

## 8. Teste com frases novas, escritas pela equipe

São frases que **não estão no CSV**, escritas como um cidadão escreveria. É o teste mais próximo de um uso real.

In [ ]:
frases_novas = [
    ("Caiu um poste na orla de Atalaia",                              "Iluminação Pública"),
    ("A lâmpada do poste da minha rua está piscando faz três dias",   "Iluminação Pública"),
    ("a rua ta um breu a noite, perigoso demais",                     "Iluminação Pública"),
    ("Tem um buraco gigante na Avenida Hermes Fontes",                "Obras e Pavimentação"),
    ("o asfalto da minha rua afundou depois da chuva",                "Obras e Pavimentação"),
    ("a ponte de madeira do canal está podre",                        "Obras e Pavimentação"),
    ("Tem lixo jogado na porta da minha casa",                        "Saneamento e Limpeza"),
    ("esgoto correndo a céu aberto no bairro",                        "Saneamento e Limpeza"),
    ("falta água aqui em casa desde ontem",                           "Saneamento e Limpeza"),
    ("O semáforo do cruzamento está desligado",                       "Trânsito e Mobilidade"),
    ("carro estacionado em cima da calçada",                          "Trânsito e Mobilidade"),
    ("o ônibus da linha do bairro demora mais de uma hora",           "Trânsito e Mobilidade"),
]

frases_do_csv = set(data_frame["Texto"])
acertos = 0

for frase, esperado in frases_novas:
    previsto = modelo_baseline.predict([frase])[0]
    confianca = modelo_baseline.predict_proba([frase]).max()
    acertou = previsto == esperado
    acertos += acertou
    marca = "OK  " if acertou else "ERRO"
    aviso = "  (atenção: frase já está no CSV)" if frase in frases_do_csv else ""
    print(f"[{marca}] {frase}\n        -> {previsto} ({confianca:.0%} de confiança){aviso}")

print(f"\nAcertos: {acertos} de {len(frases_novas)}")

**Observação:** a frase "falta água aqui em casa desde ontem" erra com confiança baixa. O dataset tem poucos exemplos de abastecimento de água, e as palavras "casa" e "ontem" aparecem em outras categorias. Mais exemplos desse tipo devem ajudar.

## 9. Demonstração: classificar um chamado

Altere a frase em `chamado_teste` e execute a célula (**Shift + Enter**). O modelo mostra a secretaria escolhida e a probabilidade de cada uma.

In [ ]:
def classificar_chamado(texto):
    probabilidades = modelo_baseline.predict_proba([texto])[0]
    categorias = modelo_baseline.classes_
    ranking = sorted(zip(categorias, probabilidades), key=lambda par: par[1], reverse=True)

    print(f"Chamado: {texto}")
    print(f"Encaminhar para: {ranking[0][0]}\n")
    for categoria, prob in ranking:
        print(f"  {categoria:<24} {prob:6.1%}")

chamado_teste = "O ônibus do meu bairro demora demais"
classificar_chamado(chamado_teste)

## Conclusões do Check-in 3

**O que funciona:**
- Pipeline completo e reprodutível: dados → limpeza → vetorização → Naive Bayes → avaliação → previsão.
- Cerca de **84% de acurácia** no conjunto de teste e cerca de **81% na validação cruzada** (média de 5 rodadas), com as quatro categorias em desempenho parecido (F1 entre 0,83 e 0,86).
- Acertou 11 de 12 frases novas escritas pela equipe.

**Limitações:**
- O dataset é **sintético**: o desempenho com reclamações reais da população ainda precisa ser verificado.
- O Naive Bayes conta palavras, mas não entende negação nem contexto.
- Algumas confianças são baixas (perto de 50%), o que indica que o modelo está em dúvida.

**Próximos passos (Check-in 4):**
- Acrescentar exemplos dos tipos que mais erram (abastecimento de água, semáforos, sinalização pintada).
- Comparar com outra representação (TF-IDF) e outro modelo.
- Encaminhar para revisão humana os chamados com confiança abaixo de um limite.
- Analisar viés e explicabilidade: quais palavras mais pesam em cada categoria.